In [ ]:
import json
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1

print('Torch version:', torch.__version__)

: 

In [ ]:
ROOT = Path('.').resolve()
PAIRS_CSV = ROOT / 'data' / 'facenet_pairs.csv'
REPORT_PATH = ROOT / 'reports' / 'facenet_report.json'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEFAULT_THRESHOLD = 0.75

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Root:', ROOT)
print('CSV:', PAIRS_CSV)
print('Device:', DEVICE)

In [ ]:
TRUE_VALUES = {'1', 'true', 'yes', 'y', 'same', 'match', 'positive'}

def parse_bool_label(raw: str) -> bool:
    return str(raw).strip().lower() in TRUE_VALUES

def safe_div(n: float, d: float) -> float:
    return 0.0 if d == 0 else n / d

class FaceNetVerifier:
    def __init__(self, device: str = 'cpu', threshold: float = 0.75, image_size: int = 160):
        self.device = device
        self.threshold = threshold
        self.mtcnn = MTCNN(
            image_size=image_size,
            margin=20,
            keep_all=False,
            post_process=True,
            device=self.device,
        )
        self.model = InceptionResnetV1(pretrained='vggface2').eval().to(self.device)

    def embedding(self, image_path: Path) -> np.ndarray:
        if not image_path.exists():
            raise FileNotFoundError(f'Image not found: {image_path}')
        image = Image.open(image_path).convert('RGB')
        face = self.mtcnn(image)
        if face is None:
            raise ValueError(f'No face detected in: {image_path}')

        with torch.no_grad():
            emb = self.model(face.unsqueeze(0).to(self.device)).squeeze(0).cpu().numpy()

        norm = np.linalg.norm(emb)
        if norm == 0:
            raise ValueError(f'Zero embedding norm in: {image_path}')
        return emb / norm

    @staticmethod
    def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-12
        return float(np.dot(a, b) / denom)

    def verify_pair(self, image_a: Path, image_b: Path, threshold: float = None) -> Dict:
        th = self.threshold if threshold is None else threshold
        ea = self.embedding(image_a)
        eb = self.embedding(image_b)
        sim = self.cosine_similarity(ea, eb)
        pred = sim >= th
        return {
            'image_a': str(image_a),
            'image_b': str(image_b),
            'similarity': float(sim),
            'threshold': float(th),
            'pred_same_person': bool(pred),
        }

verifier = FaceNetVerifier(device=DEVICE, threshold=DEFAULT_THRESHOLD)
print('FaceNet verifier initialized.')

In [ ]:
if not PAIRS_CSV.exists():
    raise FileNotFoundError(f'Missing CSV: {PAIRS_CSV}')

df = pd.read_csv(PAIRS_CSV)
required_cols = {'image_a', 'image_b', 'label'}
missing = required_cols.difference(df.columns)
if missing:
    raise ValueError(f'CSV missing required columns: {sorted(missing)}')

df['actual_same_person'] = df['label'].astype(str).map(parse_bool_label)
df.head()

In [ ]:
def evaluate_threshold(dataframe: pd.DataFrame, threshold: float) -> Dict:
    tp = tn = fp = fn = 0
    skipped = 0
    rows: List[Dict] = []

    for _, row in dataframe.iterrows():
        image_a = Path(str(row['image_a']))
        image_b = Path(str(row['image_b']))
        actual = bool(row['actual_same_person'])

        try:
            out = verifier.verify_pair(image_a=image_a, image_b=image_b, threshold=threshold)
            pred = bool(out['pred_same_person'])

            if actual and pred:
                tp += 1
            elif (not actual) and (not pred):
                tn += 1
            elif (not actual) and pred:
                fp += 1
            else:
                fn += 1

            rows.append({
                'image_a': str(image_a),
                'image_b': str(image_b),
                'actual_same_person': actual,
                'pred_same_person': pred,
                'similarity': float(out['similarity']),
                'error': ''
            })
        except Exception as exc:
            skipped += 1
            rows.append({
                'image_a': str(image_a),
                'image_b': str(image_b),
                'actual_same_person': actual,
                'pred_same_person': None,
                'similarity': None,
                'error': str(exc),
            })

    total = tp + tn + fp + fn
    accuracy = safe_div(tp + tn, total)
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * precision * recall, precision + recall)

    return {
        'threshold': float(threshold),
        'processed': int(total),
        'skipped': int(skipped),
        'tp': int(tp),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'accuracy': float(round(accuracy, 6)),
        'precision': float(round(precision, 6)),
        'recall': float(round(recall, 6)),
        'f1': float(round(f1, 6)),
        'rows': rows,
    }

threshold_grid = [round(x, 2) for x in np.arange(0.40, 0.96, 0.02)]
sweep = [evaluate_threshold(df, t) for t in threshold_grid]
sweep_summary = pd.DataFrame([{k: v for k, v in s.items() if k != 'rows'} for s in sweep])
sweep_summary.sort_values(['f1', 'accuracy'], ascending=False).head(10)

In [ ]:
best = max(sweep, key=lambda x: (x['f1'], x['accuracy']))
best_threshold = best['threshold']
print('Best threshold:', best_threshold)

final_result = evaluate_threshold(df, best_threshold)
final_summary = {k: v for k, v in final_result.items() if k != 'rows'}
final_summary

In [ ]:
report = {
    'model_info': {
        'embedding_model': 'InceptionResnetV1 pretrained=vggface2',
        'face_detector': 'MTCNN',
        'device': DEVICE
    },
    'dataset': {
        'pairs_csv': str(PAIRS_CSV),
        'rows': int(len(df))
    },
    'threshold_sweep': [{k: v for k, v in s.items() if k != 'rows'} for s in sweep],
    'selected_threshold': best_threshold,
    'final_summary': final_summary,
    'final_predictions': final_result['rows'],
}

REPORT_PATH.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Saved report to:', REPORT_PATH)

In [ ]:
# img_a = Path('./data/faces/person1_a.jpg')
# img_b = Path('./data/faces/person1_b.jpg')
# print(verifier.verify_pair(img_a, img_b, threshold=best_threshold))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Build evaluation dataframe from final predictions
viz_df = pd.DataFrame(final_result['rows'])
valid_df = viz_df[(viz_df['error'] == '') & viz_df['similarity'].notna() & viz_df['pred_same_person'].notna()].copy()

if valid_df.empty:
    raise ValueError('No valid prediction rows available for plotting. Check data paths and rerun previous cells.')

y_true = valid_df['actual_same_person'].astype(bool).to_numpy()
y_pred = valid_df['pred_same_person'].astype(bool).to_numpy()
scores = valid_df['similarity'].astype(float).to_numpy()

# --- Confusion Matrix ---
cm = np.array([
    [np.sum((~y_true) & (~y_pred)), np.sum((~y_true) & (y_pred))],
    [np.sum((y_true) & (~y_pred)), np.sum((y_true) & (y_pred))],
], dtype=int)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Pred: Different', 'Pred: Same'],
    yticklabels=['Actual: Different', 'Actual: Same'],
)
plt.title(f'FaceNet Confusion Matrix @ threshold={best_threshold:.2f}')
plt.tight_layout()
plt.show()

# --- ROC Curve (manual threshold sweep) ---
thresholds = np.unique(scores)
if thresholds.size == 1:
    thresholds = np.array([thresholds[0] - 1e-6, thresholds[0], thresholds[0] + 1e-6])

# include extremes to force full curve endpoints
thresholds = np.concatenate(([scores.max() + 1e-6], np.sort(thresholds)[::-1], [scores.min() - 1e-6]))

roc_points = []
for t in thresholds:
    yp = scores >= t
    tp = np.sum((y_true) & (yp))
    tn = np.sum((~y_true) & (~yp))
    fp = np.sum((~y_true) & (yp))
    fn = np.sum((y_true) & (~yp))

    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    roc_points.append((fpr, tpr))

roc_points = sorted(roc_points, key=lambda x: x[0])
fpr_vals = np.array([p[0] for p in roc_points])
tpr_vals = np.array([p[1] for p in roc_points])
auc = float(np.trapz(tpr_vals, fpr_vals))

plt.figure(figsize=(6, 5))
plt.plot(fpr_vals, tpr_vals, label=f'FaceNet ROC (AUC={auc:.4f})', color='tab:blue')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Face Verification ROC Curve')
plt.legend(loc='lower right')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print(f'Valid rows used for plotting: {len(valid_df)}')
print(f'ROC AUC: {auc:.6f}')

## Advanced Evaluation: Operating Point, Confidence Intervals, and Error Analysis

This section adds stronger evidence for research-style reporting:
- compares operating points across thresholds (F1 and Youden's J)
- estimates uncertainty using bootstrap confidence intervals
- inspects high-impact false positives and false negatives

In [ ]:
from numpy.random import default_rng


def _f1_from_counts(tp: int, fp: int, fn: int) -> float:
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    return safe_div(2 * precision * recall, precision + recall)


def _roc_auc_manual(y_true_bool: np.ndarray, score_vals: np.ndarray) -> float:
    thresholds = np.unique(score_vals)
    if thresholds.size == 1:
        thresholds = np.array([thresholds[0] - 1e-6, thresholds[0], thresholds[0] + 1e-6])
    thresholds = np.concatenate(([score_vals.max() + 1e-6], np.sort(thresholds)[::-1], [score_vals.min() - 1e-6]))

    points = []
    for t in thresholds:
        yp = score_vals >= t
        tp = np.sum((y_true_bool) & (yp))
        tn = np.sum((~y_true_bool) & (~yp))
        fp = np.sum((~y_true_bool) & (yp))
        fn = np.sum((y_true_bool) & (~yp))
        tpr = safe_div(tp, tp + fn)
        fpr = safe_div(fp, fp + tn)
        points.append((fpr, tpr))

    points = sorted(points, key=lambda x: x[0])
    fpr_vals = np.array([p[0] for p in points])
    tpr_vals = np.array([p[1] for p in points])
    return float(np.trapz(tpr_vals, fpr_vals))


def _bootstrap_ci(values: np.ndarray, alpha: float = 0.05) -> tuple:
    clean = values[np.isfinite(values)]
    if clean.size == 0:
        return (float('nan'), float('nan'), float('nan'))
    return (
        float(np.quantile(clean, alpha / 2)),
        float(np.quantile(clean, 0.5)),
        float(np.quantile(clean, 1 - alpha / 2)),
    )


# --- Operating point diagnostics from sweep ---
ops_df = sweep_summary.copy()
ops_df['fpr'] = ops_df.apply(lambda r: safe_div(r['fp'], r['fp'] + r['tn']), axis=1)
ops_df['youden_j'] = ops_df['recall'] - ops_df['fpr']

cols = ['threshold', 'accuracy', 'precision', 'recall', 'f1', 'fpr', 'youden_j']
print('Top thresholds by F1')
print(ops_df.sort_values(['f1', 'accuracy'], ascending=False)[cols].head(5).to_string(index=False))
print('\nTop thresholds by Youden J')
print(ops_df.sort_values(['youden_j', 'accuracy'], ascending=False)[cols].head(5).to_string(index=False))

# --- Bootstrap confidence intervals at selected threshold ---
rng = default_rng(42)
selected_pred = scores >= float(best_threshold)
n = len(y_true)

boot_acc, boot_f1, boot_auc = [], [], []
for _ in range(1000):
    idx = rng.integers(0, n, n)
    yt = y_true[idx]
    yp = selected_pred[idx]
    sc = scores[idx]

    tp = int(np.sum((yt) & (yp)))
    tn = int(np.sum((~yt) & (~yp)))
    fp = int(np.sum((~yt) & (yp)))
    fn = int(np.sum((yt) & (~yp)))

    boot_acc.append(safe_div(tp + tn, tp + tn + fp + fn))
    boot_f1.append(_f1_from_counts(tp, fp, fn))

    if np.unique(yt).size > 1:
        boot_auc.append(_roc_auc_manual(yt, sc))

ci_table = pd.DataFrame(
    {
        'metric': ['accuracy', 'f1', 'roc_auc'],
        'ci_95_lower': [
            _bootstrap_ci(np.array(boot_acc))[0],
            _bootstrap_ci(np.array(boot_f1))[0],
            _bootstrap_ci(np.array(boot_auc))[0],
        ],
        'median': [
            _bootstrap_ci(np.array(boot_acc))[1],
            _bootstrap_ci(np.array(boot_f1))[1],
            _bootstrap_ci(np.array(boot_auc))[1],
        ],
        'ci_95_upper': [
            _bootstrap_ci(np.array(boot_acc))[2],
            _bootstrap_ci(np.array(boot_f1))[2],
            _bootstrap_ci(np.array(boot_auc))[2],
        ],
    }
)

print('\nBootstrap 95% confidence intervals (n=1000)')
print(ci_table.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# --- Error analysis ---
analysis_df = valid_df.copy()
analysis_df['error_type'] = np.where(
    (analysis_df['actual_same_person'] == False) & (analysis_df['pred_same_person'] == True),
    'false_positive',
    np.where(
        (analysis_df['actual_same_person'] == True) & (analysis_df['pred_same_person'] == False),
        'false_negative',
        'correct',
    ),
)

errors_only = analysis_df[analysis_df['error_type'] != 'correct'].copy()
print(f'\nTotal errors: {len(errors_only)} / {len(analysis_df)}')
if not errors_only.empty:
    print(errors_only['error_type'].value_counts().to_string())

    fp_top = errors_only[errors_only['error_type'] == 'false_positive'].sort_values('similarity', ascending=False).head(8)
    fn_top = errors_only[errors_only['error_type'] == 'false_negative'].sort_values('similarity', ascending=True).head(8)

    if not fp_top.empty:
        print('\nMost confident false positives (possible look-alikes):')
        print(fp_top[['image_a', 'image_b', 'similarity']].to_string(index=False))

    if not fn_top.empty:
        print('\nMost severe false negatives (likely missed same-person pairs):')
        print(fn_top[['image_a', 'image_b', 'similarity']].to_string(index=False))